Summary of EDA findings:

Dataset covers 20 countries from 2020 - 2025 with a binary target variable (conflict-escalation_6m). EDA raised concerns regarding the variables real-world validity, showing erratic month-to-month behavior and missing major events such as the USA's Jan 6 Insurrection/Riots, while accurately predicting conflicts like the Russia-Ukraine war, a month prior.

Most features are continuous numeric floats with outliers present. Dropped regime_type, election_cycle coumns for unclear or low signal. Dropped political_stability_index due to multicollinearity with instability score. 

Predictive signals identified include arms imports, social media sentiment, and standard economic stress indicators like inflationa and unemployment. 

Brief: 

Given the availability of multiple continuous variables, I had initially thought that logistic regression might make a good fit to model this data with. However, I'm now thinking that logistic regression will make a better baseline model and that I would like to opt for Gradient Boosting instead. 

In [1]:
#depenencies, settings & reimport of treated files
import os 
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from IPython.display import display
import seaborn as sns
import sklearn.decomposition as decomp 
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
pd.set_option('display.max_columns', None)

# reimport treated files
dataset = pd.read_csv("dataset_cleaned.csv")

dataset.head()


,country,region,month,gdp_growth_pct,inflation_rate,unemployment_rate,food_price_index,energy_dependency_pct,military_expenditure_pct_gdp,arms_imports_index,border_disputes_count,refugee_outflow_thousands,sanctions_active,media_freedom_score,protest_events_last_3m,cyber_attack_incidents,last_conflict_year,trade_dependency_rival_pct,foreign_troops_present,social_media_sentiment,rolling_protest_avg_6m,instability_score,conflict_escalation_6m
0,USA,North America,2020-01-01,6.737638,5.308678,9.069739,108.263140,60.851201,2.536582,28.356756,3,52.666619,0,45.990963,3,7,2018-01-01,22.210790,1,-0.907099,2.144414,52.436131,0
1,USA,North America,2020-02-01,4.467635,7.878490,1.116625,125.221590,10.806598,1.671814,44.921531,1,76.236598,0,79.921825,8,5,2018-01-01,48.252741,0,-0.457302,9.763522,98.474060,1
2,USA,North America,2020-03-01,-0.030766,11.285611,8.446705,145.774988,68.625602,2.160782,32.269691,0,130.978395,0,24.131293,4,8,2018-01-01,31.608400,1,0.122487,6.199294,80.755509,1
3,USA,North America,2020-04-01,0.155373,5.594405,7.331570,85.482074,44.383874,2.940183,64.029344,4,158.423899,0,36.203995,4,9,2018-01-01,47.428411,0,0.389570,4.038001,58.176382,0
4,USA,North America,2020-05-01,0.755706,10.483152,0.405036,111.397986,65.015718,2.672375,18.460049,5,19.204378,0,47.762006,2,9,2018-01-01,17.123555,1,-0.663418,1.264349,60.718597,0


Roadmap: 

I think I want to run a PCA log regression model to establish a baseline, then treat data for a gradient boosting model. 

I'm going to start by adding additional features. 

Splitting a train/test split. 

Then, building a pipeline - scaling and transforming. 

Then, running PCA & log regression. 

I'll use Gradient Boosting in the next Capstone Step. 

In [2]:
dataset['month'] = pd.to_datetime(dataset['month'], errors='coerce')
dataset['last_conflict_year'] = pd.to_datetime(dataset['last_conflict_year'], errors='coerce')

print(dataset['month'].isnull().sum())
print(dataset['last_conflict_year'].isnull().sum())

0
0


In [3]:
#feature engineering

dataset = dataset.sort_values(['country', 'month'])

rolling_features = ['arms_imports_index',
                    'gdp_growth_pct',
                    'inflation_rate',
                    'unemployment_rate',
                    'food_price_index',
                    'social_media_sentiment']

for col in rolling_features:
    dataset[f'{col}_roll3'] = dataset.groupby('country')[col].transform(lambda x: x.rolling(3).mean())
    dataset[f'{col}_roll6'] = dataset.groupby('country')[col].transform(lambda x: x.rolling(6).mean())
    dataset[f'{col}_lag1'] = dataset.groupby('country')[col].shift(1)
    dataset[f'{col}_lag3'] = dataset.groupby('country')[col].shift(3)

dataset['years_since_last_conflict'] = (dataset['month'].dt.year - dataset['last_conflict_year'].dt.year)

dataset = pd.get_dummies(dataset, columns=['region'], drop_first=True)

dataset.head()

,country,month,gdp_growth_pct,inflation_rate,unemployment_rate,food_price_index,energy_dependency_pct,military_expenditure_pct_gdp,arms_imports_index,border_disputes_count,refugee_outflow_thousands,sanctions_active,media_freedom_score,protest_events_last_3m,cyber_attack_incidents,last_conflict_year,trade_dependency_rival_pct,foreign_troops_present,social_media_sentiment,rolling_protest_avg_6m,instability_score,conflict_escalation_6m,arms_imports_index_roll3,arms_imports_index_roll6,arms_imports_index_lag1,arms_imports_index_lag3,gdp_growth_pct_roll3,gdp_growth_pct_roll6,gdp_growth_pct_lag1,gdp_growth_pct_lag3,inflation_rate_roll3,inflation_rate_roll6,inflation_rate_lag1,inflation_rate_lag3,unemployment_rate_roll3,unemployment_rate_roll6,unemployment_rate_lag1,unemployment_rate_lag3,food_price_index_roll3,food_price_index_roll6,food_price_index_lag1,food_price_index_lag3,social_media_sentiment_roll3,social_media_sentiment_roll6,social_media_sentiment_lag1,social_media_sentiment_lag3,years_since_last_conflict,region_Eastern Europe,region_Middle East,region_North America,region_South America,region_South Asia,region_Western Europe
1254,Brazil,2020-01-01,3.292274,4.821969,4.291811,105.628769,71.769859,3.089986,92.141774,4,132.543939,0,27.543265,0,13,2014-01-01,29.655392,1,0.279130,2.177743,39.090019,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6,False,False,False,True,False,False
1255,Brazil,2020-02-01,5.265491,9.813665,0.620002,129.194697,54.527504,1.009883,17.864491,3,151.380973,0,42.125575,8,16,2018-01-01,25.698183,0,0.730459,8.768982,92.156494,1,NaN,NaN,92.141774,NaN,NaN,NaN,3.292274,NaN,NaN,NaN,4.821969,NaN,NaN,NaN,4.291811,NaN,NaN,NaN,105.628769,NaN,NaN,NaN,0.279130,NaN,2,False,False,False,True,False,False
1256,Brazil,2020-03-01,-0.974037,10.701998,6.914191,130.586562,39.088637,3.652373,51.928079,2,153.135730,0,84.333469,1,14,2018-01-01,11.518326,0,0.541363,2.242119,64.168321,0,53.978115,NaN,17.864491,NaN,2.527909,NaN,5.265491,NaN,8.445877,NaN,9.813665,NaN,3.942001,NaN,0.620002,NaN,121.803343,NaN,129.194697,NaN,0.516984,NaN,0.730459,NaN,2,False,False,False,True,False,False
1257,Brazil,2020-04-01,-1.595410,10.456867,12.601752,146.127866,93.185635,1.480844,31.218015,2,16.774349,0,25.417723,15,15,2018-01-01,20.570074,0,-0.956771,15.218694,124.992587,1,33.670195,NaN,51.928079,92.141774,0.898681,NaN,-0.974037,3.292274,10.324177,NaN,10.701998,4.821969,6.711981,NaN,6.914191,4.291811,135.303042,NaN,130.586562,105.628769,0.105017,NaN,0.541363,0.279130,2,False,False,False,True,False,False
1258,Brazil,2020-05-01,0.313102,6.699544,8.975298,99.876591,14.000469,4.088039,28.890343,3,211.967334,0,79.535580,4,9,2018-01-01,16.016826,1,-0.420020,1.643895,58.845819,0,37.345479,NaN,31.218015,17.864491,-0.752115,NaN,-1.595410,5.265491,9.286136,NaN,10.456867,9.813665,9.497080,NaN,12.601752,0.620002,125.530340,NaN,146.127866,129.194697,-0.278476,NaN,-0.956771,0.730459,2,False,False,False,True,False,False


In [4]:
# Export cleaned dataset for pre-processing notebook

output_dir = r"C:\Users\ryan1\Desktop\Capstone Two Project Files"

# Main cleaned dataset
dataset.to_csv(os.path.join(output_dir, "dataset_data_pre_processed.csv"), index=False)

print("Export complete.")
print(f"Shape: {dataset.shape}")
print(f"Columns: {list(dataset.columns)}")

Export complete.
Shape: (1320, 53)
Columns: ['country', 'month', 'gdp_growth_pct', 'inflation_rate', 'unemployment_rate', 'food_price_index', 'energy_dependency_pct', 'military_expenditure_pct_gdp', 'arms_imports_index', 'border_disputes_count', 'refugee_outflow_thousands', 'sanctions_active', 'media_freedom_score', 'protest_events_last_3m', 'cyber_attack_incidents', 'last_conflict_year', 'trade_dependency_rival_pct', 'foreign_troops_present', 'social_media_sentiment', 'rolling_protest_avg_6m', 'instability_score', 'conflict_escalation_6m', 'arms_imports_index_roll3', 'arms_imports_index_roll6', 'arms_imports_index_lag1', 'arms_imports_index_lag3', 'gdp_growth_pct_roll3', 'gdp_growth_pct_roll6', 'gdp_growth_pct_lag1', 'gdp_growth_pct_lag3', 'inflation_rate_roll3', 'inflation_rate_roll6', 'inflation_rate_lag1', 'inflation_rate_lag3', 'unemployment_rate_roll3', 'unemployment_rate_roll6', 'unemployment_rate_lag1', 'unemployment_rate_lag3', 'food_price_index_roll3', 'food_price_index_roll6

In [5]:
#Pre Processing 

dataset = dataset.drop(columns=['country', 'last_conflict_year'])

train = dataset[dataset['month'] < '2024-01-01']
test = dataset[dataset['month'] >= '2024-01-01']

train = train.drop(columns=['month'])
test = test.drop(columns=['month'])

X_train = train.drop(columns=['conflict_escalation_6m'])
y_train = train['conflict_escalation_6m']

X_test = test.drop(columns=['conflict_escalation_6m'])
y_test = test['conflict_escalation_6m']


In [8]:
#pipeline for scaling and PCA

X_train = X_train.dropna()
y_train = y_train[X_train.index]

X_test = X_test.dropna()
y_test = y_test[X_test.index]

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=0.95)),
    ('model', LogisticRegression())
])

pipeline.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()), ('pca', PCA(n_components=0.95)),
                ('model', LogisticRegression())])

In [9]:
y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.86      0.92      0.89       250
           1       0.78      0.65      0.71       110

    accuracy                           0.84       360
   macro avg       0.82      0.79      0.80       360
weighted avg       0.84      0.84      0.83       360

[[230  20]
 [ 38  72]]


Over the course of this module, I added features to the time series dataset which would preserve time-series relationships inside of a classification model. I then split my dataset chronologically, before applying scaling and PCA to reduce dimensionality and run it through a multivariate logistic regression model. 

The model which will serve as a baseline for future modelling efforts showed an impressive class 0 recall, wiuth a much lower class 1 recall. In other words, it wasn't very good at predicting 6m conflict escalations accurately -- which was the entire goal of the model. 

This baseline provides ample opportunity for improvement thorugh more advanced modeling efforts. 